# Full configuration interaction theory

Companion notebook to Chapter 5 of *Quantum mechanics for many-particle
systems*.  Every number quoted in the chapter is produced here.  The code is
the same as in `BookManybody/BookMaterial/Programs/fci.py`; the notebook runs
it section by section.

Contents:

1. Slater determinants as bit strings, and the excitation-level classification
2. The pairing model: full space versus the seniority-zero subspace
3. The block structure of the Hamiltonian matrix
4. Truncations: CIS, CID, CISD and full CI
5. Size consistency, and why coupled cluster repairs it
6. The Hubbard ring in the momentum basis
7. The exponential wall

## 1. Setup

We import the module directly, so that the notebook and the chapter can never
drift apart.  Adjust the path if you have moved the programs.

In [1]:
import sys, os
import numpy as np

sys.path.insert(0, os.path.join("..", "BookManybody", "BookMaterial", "Programs"))
import fci

np.set_printoptions(precision=6, suppress=True, linewidth=120)

### Determinants as bit strings

A Slater determinant with $N$ particles in $n$ single-particle states is an
integer with $N$ bits set, exactly as in the last section of Chapter 3.
Creation and annihilation act by setting and clearing bits, and the sign is
$(-1)^{\ell}$ with $\ell$ the number of set bits below the one acted on.

In [2]:
basis = fci.SlaterBasis(8, 4)          # 8 spin-orbitals, 4 particles
print("dimension:", basis.dim, "= C(8,4)")
for s in basis.states[:5]:
    print(format(s, "08b"), "  occupied:", [k for k in range(8) if s >> k & 1])

dimension: 70 = C(8,4)
00001111   occupied: [0, 1, 2, 3]
00010111   occupied: [0, 1, 2, 4]
00011011   occupied: [0, 1, 3, 4]
00011101   occupied: [0, 2, 3, 4]
00011110   occupied: [1, 2, 3, 4]


In [3]:
state = 0b00101101
for p in (1, 4):
    sign, new = fci.create(state, p)
    if sign:
        print(f"a+_{p} |{state:08b}> = {sign:+d} |{new:08b}>")
    else:
        print(f"a+_{p} |{state:08b}> = 0   (orbital already occupied)")

a+_1 |00101101> = -1 |00101111>
a+_4 |00101101> = -1 |00111101>


### Excitation levels

Classifying every determinant by how many particles sit outside the reference
gives the distribution $1, 16, 36, 16, 1$ predicted by
$\binom{N}{k}\binom{n-N}{k}$ -- Exercise 1 of the chapter.

In [4]:
reference = basis.states[0]
levels = basis.excitation_levels(reference)
for k in range(5):
    print(f"{k}p-{k}h: {(levels == k).sum():3d}")
print("total  :", basis.dim)

0p-0h:   1
1p-1h:  16
2p-2h:  36
3p-3h:  16
4p-4h:   1
total  : 70


## 2. The pairing model

$$
\hat H = \xi\sum_{p\sigma}(p-1)\,a^{\dagger}_{p\sigma}a_{p\sigma}
      - \tfrac12 g\sum_{pq} a^{\dagger}_{p+}a^{\dagger}_{p-}a_{q-}a_{q+}
$$

Four doubly degenerate levels, four particles, $\xi = 1$.  The full space has
$\binom{8}{4} = 70$ determinants; the seniority-zero subspace used in
Chapter 4 has only $\binom{4}{2} = 6$.  Both give the same ground-state
energy.

In [5]:
fci.demo_full_space()

1. The pairing model in the full CI space
four levels, four particles: 8 spin-orbitals,
dimension of the full space  = 70 = C(8,4)
seniority-zero subspace      = 6 (chapter 4 worked here)

determinants by excitation level relative to the reference:
   0p-0h:    1
   1p-1h:   16
   2p-2h:   36
   3p-3h:   16
   4p-4h:    1


In [6]:
model = fci.PairingFCI(levels=4, n_particles=4, g=1.0, xi=1.0)
w, v = np.linalg.eigh(model.matrix())
print("ground-state energy, full 70-dimensional space :", f"{w[0]:.8f}")
print("Table 4.2, seniority-zero subspace, g = 1      :  0.63554847")

psi = v[:, 0]
odd = np.where(model.levels_of % 2 == 1)[0]
print("\nlargest amplitude on a broken-pair determinant :",
      f"{np.abs(psi[odd]).max():.2e}")

ground-state energy, full 70-dimensional space : 0.63554847
Table 4.2, seniority-zero subspace, g = 1      :  0.63554847

largest amplitude on a broken-pair determinant : 1.11e-16


## 3. The block structure

A two-body operator connects determinants differing by at most two
single-particle states, so the matrix is banded in the excitation level.  For
the pairing model the seniority symmetry empties the odd blocks as well, and
the matrix is banded with a gap.

In [7]:
fci.demo_block_structure()

2. The block structure of the Hamiltonian matrix
        0p-0h  1p-1h  2p-2h  3p-3h  4p-4h  
 0p-0h    x      0      x      0      0    
 1p-1h    0      x      0      x      0    
 2p-2h    x      0      x      0      x    
 3p-3h    0      x      0      x      0    
 4p-4h    0      0      x      0      x    

A two-body operator connects determinants differing by at most
two single-particle states, so the matrix is banded in the
excitation level: the Condon-Slater rules of chapter 2 read as a
statement about sparsity.


## 4. Truncations

Keeping all determinants up to $n$ particle-hole pairs gives CIS, CID, CISD
and so on.  Each is a variational calculation in a subspace, so each energy is
an upper bound to the exact one, and the bounds fall monotonically as the
truncation is relaxed.

For the pairing model the singles do nothing at all -- the interaction cannot
break a pair -- and CISD coincides with CID.

In [8]:
fci.demo_truncations()

3. Truncating the CI expansion
     g       E_ref           CIS           CID          CISD           FCI
  0.25    1.750000    1.75000000    1.73050700    1.73050700    1.73045985
  0.50    1.500000    1.50000000    1.41759645    1.41759645    1.41677428
  1.00    1.000000    1.00000000    0.64890655    0.64890655    0.63554847
  2.00    0.000000    0.00000000   -1.35208670   -1.35208670   -1.48965216

basis sizes: 0p-0h and below:   1   1p-1h and below:  17   2p-2h and below:  53   3p-3h and below:  69   4p-4h and below:  70   full: 70

Singles alone change nothing: the pairing interaction cannot
break a pair, so <Phi_0|H|Phi_i^a> = 0 and CIS = reference.
Doubles capture almost all of the correlation energy, and CISD
equals CID here for the same reason.


## 5. Size consistency

Two identical non-interacting subsystems must give exactly twice the energy of
one.  Full CI does; CID does not, because two simultaneous double excitations
form a quadruple excitation of the combined reference.

This is the defect that coupled-cluster theory repairs, by writing
$e^{\hat T_2}$ instead of $1 + \hat C_2$: the term
$\tfrac12 \hat T_2^2$ then appears automatically, with its coefficient fixed
by the doubles rather than determined independently.

In [9]:
fci.demo_size_consistency()

4. Size consistency
One subsystem: two levels, two particles.  Then two identical
copies with no interaction between them.  A size-consistent
method must give exactly twice the energy.

     g          E(A)        2 E(A)         E(AB)        error   method
  0.25   -0.13278222   -0.26556444   -0.26556444     0.00e+00   FCI
  0.25   -0.13278222   -0.26556444   -0.26550480     5.96e-05   CID
  0.50   -0.28077641   -0.56155281   -0.56155281     0.00e+00   FCI
  0.50   -0.28077641   -0.56155281   -0.56066017     8.93e-04   CID
  1.00   -0.61803399   -1.23606798   -1.23606798    -2.22e-16   FCI
  1.00   -0.61803399   -1.23606798   -1.22474487     1.13e-02   CID

Full CI passes to machine precision.  CID does not: for the
combined system, both subsystems being doubly excited at once is
a quadruple excitation, and CID has no room for it.  The error
grows with the number of subsystems, so truncated CI becomes
useless for large systems.  This is the failure that
coupled-cluster theory repairs b

## 6. The Hubbard ring in the momentum basis

$$
\hat H = \sum_{k\sigma}\varepsilon_k\,c^{\dagger}_{k\sigma}c_{k\sigma}
 + \frac{U}{L}\sum_{kk'q} c^{\dagger}_{k+q\uparrow}c^{\dagger}_{k'-q\downarrow}
   c_{k'\downarrow}c_{k\uparrow},
\qquad \varepsilon_k = -2t\cos k .
$$

Six sites at half filling, dimension $\binom{6}{3}^2 = 400$.  Watch the
fraction of the correlation energy recovered by the doubles fall as $U/t$
grows: this is the onset of strong correlation, and it is where every method
built on a single reference determinant begins to fail.

In [10]:
fci.demo_hubbard()

5. The Hubbard ring in the momentum basis
   U/t        E_ref           CIS           CID          CISD           FCI  % recovered


   1.0    -6.500000   -6.50000000   -6.59980159   -6.59989778   -6.60115829       98.66%


   2.0    -5.000000   -5.00000000   -5.38877703   -5.39026458   -5.40945685       94.95%


   4.0    -2.000000   -2.00000000   -3.41192866   -3.43018837   -3.66870618       84.61%


   8.0     4.000000    2.48338852   -0.36779217   -1.05692183   -2.04813089       72.22%

six sites at half filling: dimension 400 = C(6,3)^2
determinants by excitation level: 0p-0h: 1  1p-1h: 18  2p-2h: 99  3p-3h: 164  4p-4h: 99  5p-5h: 18  6p-6h: 1

As U grows the reference determinant becomes a worse starting
point and the doubles recover a smaller fraction of the
correlation energy -- the signature of strong correlation.


### Why CIS equals the reference, until it does not

Total crystal momentum is conserved, so a single excitation cannot be reached
from the Fermi sea: $\langle\Phi_0|\hat H|\Phi_i^a\rangle = 0$ for every $U$.
Brillouin's theorem here follows from a symmetry rather than from a
variational condition -- for a translationally invariant problem the
plane-wave determinant *is* the Hartree-Fock solution.

The CIS matrix is therefore block diagonal, and its lowest eigenvalue is the
smaller of the reference energy and the lowest eigenvalue of the singles
block.  At $U/t = 8$ the second one wins, and the "CIS ground state" is a
state orthogonal to the reference: a legitimate variational bound that no
longer describes the state we set out to compute.

In [11]:
for U in (1.0, 2.0, 4.0, 8.0):
    h = fci.HubbardFCI(sites=6, n_up=3, n_down=3, t=1.0, U=U)
    M = h.matrix()
    r = h.index[h.reference]
    singles = np.where(h.levels_of == 1)[0]
    coupling = np.abs(M[r, singles]).max()
    lowest = np.linalg.eigvalsh(M[np.ix_(singles, singles)])[0]
    print(f"U/t = {U:4.1f}   max |<Phi_0|H|Phi_i^a>| = {coupling:8.2e}"
          f"   E_ref = {M[r, r]:9.6f}"
          f"   lowest singles eigenvalue = {lowest:9.6f}")

U/t =  1.0   max |<Phi_0|H|Phi_i^a>| = 0.00e+00   E_ref = -6.500000   lowest singles eigenvalue = -4.858678
U/t =  2.0   max |<Phi_0|H|Phi_i^a>| = 0.00e+00   E_ref = -5.000000   lowest singles eigenvalue = -3.758306


U/t =  4.0   max |<Phi_0|H|Phi_i^a>| = 0.00e+00   E_ref = -2.000000   lowest singles eigenvalue = -1.632993
U/t =  8.0   max |<Phi_0|H|Phi_i^a>| = 0.00e+00   E_ref =  4.000000   lowest singles eigenvalue =  2.483389


## 7. The exponential wall

$\dim\mathcal H = \binom{n}{N}$, and at half filling
$\binom{n}{n/2} \sim 2^n/\sqrt{n}$: the dimension doubles with every added
single-particle state.  Direct diagonalisation reaches about $10^5$, Lanczos
about $10^{10}$.  Beyond that the expansion itself must be truncated -- which
is what the remaining chapters are about.

In [12]:
fci.demo_exponential_wall()

6. The exponential wall
                          system   n_sp    N        dimension
                       toy model     10    5        2.520e+02
               small shell model     20   10        1.848e+05
    oxygen-16 neutrons, 4 shells     40    8        7.690e+07
         half filling, 40 states     40   20        1.378e+11
         half filling, 80 states     80   40        1.075e+23

At half filling the binomial coefficient becomes exponential:
     n        C(n, n/2)    2^n / sqrt(n)
    10        2.520e+02        3.238e+02
    20        1.848e+05        2.345e+05
    40        1.378e+11        1.738e+11
    80        1.075e+23        1.352e+23
   160        9.205e+46        1.155e+47

Oxygen-16 in the four lowest major shells (0s, 0p, 1s0d, 1p0f)
has 40 single-particle states for each species, so the eight
neutrons give C(40,8) = 7.690e+07 determinants and the full
proton-neutron space 5.914e+15.  Symmetries reduce this,
but adding one more shell or relaxing the truncation 

### The same wall, seen from four sides

| system | counting | dimension |
|---|---|---|
| Lipkin, $N$ particles, two levels | $\binom{2N}{N}$, or $\binom{N}{N/2}$ in the top multiplet | $\sim 2^N/\sqrt N$ |
| pairing, $n$ levels, $N$ particles | $\binom{2n}{N}$, or $\binom{n}{N/2}$ at seniority zero | exponential |
| Hubbard, $L$ sites | $\binom{L}{N_\uparrow}\binom{L}{N_\downarrow}$, at most $4^L$ | exponential |
| spin chain, $L$ sites | $\bigotimes_{i=1}^L\mathbb C^2$ | $2^L$ |

The tensor product is the fundamental statement; the binomial coefficient is
what remains after a conservation law has been imposed.  A symmetry removes a
factor, never the exponential.

What saves us is that physical ground states are not arbitrary vectors in
$\mathcal H$.  For gapped local Hamiltonians the entanglement across a cut
obeys an area law, the Schmidt spectrum decays quickly, and a matrix product
state with a modest bond dimension suffices.  The cell below measures the
Schmidt spectrum of the pairing ground state across a cut that separates the
two lowest levels from the two highest.

In [13]:
model = fci.PairingFCI(levels=4, n_particles=4, g=1.0, xi=1.0)
w, v = np.linalg.eigh(model.matrix())
psi = v[:, 0]

occ = np.array([[(s >> k) & 1 for k in range(8)] for s in model.basis.states])
labels_A = [tuple(row[:4]) for row in occ]     # levels 1 and 2
labels_B = [tuple(row[4:]) for row in occ]     # levels 3 and 4
iA = {l: i for i, l in enumerate(sorted(set(labels_A)))}
iB = {l: i for i, l in enumerate(sorted(set(labels_B)))}

M = np.zeros((len(iA), len(iB)))
for c, (a, b) in enumerate(zip(labels_A, labels_B)):
    M[iA[a], iB[b]] = psi[c]

s = np.linalg.svd(M, compute_uv=False)
s = s[s > 1e-12]
print("Schmidt coefficients:", np.round(s, 6))
p = s**2
print("entanglement entropy:", f"{-np.sum(p * np.log2(p)):.6f}", "bits")
print("\nOnly a handful of Schmidt states carry any weight -- which is exactly")
print("what makes a compressed representation possible.")

Schmidt coefficients: [0.931471 0.361414 0.040586 0.00977 ]
entanglement entropy: 0.577799 bits

Only a handful of Schmidt states carry any weight -- which is exactly
what makes a compressed representation possible.


## The full program

Everything above lives in `BookManybody/BookMaterial/Programs/fci.py`, which
runs as a script and prints all six demonstrations of the chapter.

In [14]:
print(open(fci.__file__).read())

"""
Full configuration interaction, and what happens when it is truncated.

Companion code to chapter 5 of *Quantum mechanics for Many-particle Systems*.

The Hamiltonian is built in the basis of all Slater determinants that can be
formed from a truncated set of single-particle states, and diagonalised.  Every
determinant is classified by its excitation level relative to a reference, so
that the block structure of the matrix can be displayed and the truncated
schemes -- CIS, CID, CISD and so on -- obtained by restricting the basis.

    SlaterBasis      -- determinants as bit strings, with fermionic phases
    PairingFCI       -- the pairing model in the full space, not just the
                        seniority-zero subspace
    HubbardFCI       -- the Hubbard ring in the momentum basis, where the
                        free-fermion determinant is the natural reference
    hilbert_growth   -- how the dimension explodes

Author: Morten Hjorth-Jensen
"""

from itertools import combinat